# Parser test — Qwen3.8-27B writes the verification plan

For each question:

1. **Retrieval** (the VINE graph and the vector store) returns the provisions — `retrieve_for_compile`, exactly what VINE uses.
2. **Qwen3.8-27B** reads the question and everything retrieval returned, and writes the plan: the obligations, how they depend on each other, the guards, the merges, and where each one's evidence is.
3. **Checks, no model:** the plan's structure (§3.1 of the paper) and its references to the manual. A plan that fails is sent back to the model with the problems, up to 3 times. There is no fallback here: a failure is reported as a failure.

It stops there. Nothing is executed and no answer is written. **The plan is what is being tested.**

**What the model is told** (`mrag/vine/parser.py`): the paper's own description of compilation (§3.1), the manual's own definitions of Standard, Guidance, Option and Support (Section 1C.01), and how the checkers and merges work. **Nothing from any test question or gold answer.**

**Two sets of questions**

| set | what | used for |
|---|---|---|
| DEV | 11 cases written from random MUTCD provisions (`evaluation/retrieval_dev/manual_cases_v1.jsonl`), picked by a fixed rule | looking at how the parser behaves |
| TEST | the 10 sample questions, text only | run **once**, in the last cell |

No gold answers, obligation lists or counts are anywhere in this notebook. The test plans are saved for scoring outside the pipeline.

**Runtime: A100** (40 GB or 80 GB). About 2 hours in all. Every model reply is cached on Drive, so if the runtime stops, re-running skips what is done.

**Run CELL 1 to CELL 12 in order.** Stop after CELL 9 and read the plan it prints.

If the runtime restarts after CELL 5: run CELL 1, CELL 4, CELL 5, then CELL 6 onward. (CELL 5 reads the saved retrieval results from Drive.)

In [ ]:
# CELL 1 — Drive, repo, packages.
# Same layout as the other notebooks: data on Drive, repo on local disk.
# MRAG_BASE_DIR must be set BEFORE any mrag import.
import sys, os, subprocess
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/evaluation")

# The retrieval stack, pinned as in Retrieval_Test. vLLM is NOT installed here:
# it gets its own environment in CELL 7, so these versions stay as they are.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"

for f in ("mrag/vine/vllm_client.py", "evaluation/parser_check.py", "mrag/kg_vine.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"{f} is not in the clone -- push it to GitHub, then re-run this cell"
assert "PARSER_TASK" in open(f"{REPO_DIR}/mrag/vine/parser.py").read(), (
    "mrag/vine/parser.py is the old version -- push the new one, then re-run this cell")

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("data  :", DRIVE_DIR)
print("code  :", REPO_DIR, "| commit", COMMIT)

In [ ]:
# CELL 2 — the vector store on local disk (restored from Drive if needed).
import shutil, time
from pathlib import Path
from mrag.config import CFG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = Path(CFG.base_dir) / "qdrant_db.tar"

def has_store(p: Path) -> bool:
    return (p / "collection" / CFG.coll_chunks).exists()

if has_store(LOCAL):
    print("vector store already on local disk:", LOCAL)
elif SNAP.exists():
    print(f"restoring {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) ...")
    t0 = time.time()
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)
    print(f"  done in {time.time() - t0:.0f}s")
else:
    raise FileNotFoundError(f"No vector store at {LOCAL} and no snapshot at {SNAP}. "
                            "Run notebooks/Fresh_Build.ipynb first.")
assert has_store(LOCAL), f"{LOCAL}/collection/{CFG.coll_chunks} is missing after the restore"
print("ok:", LOCAL)

In [ ]:
# CELL 3 — load the retriever (text encoder, reranker, VINE graph). No LLM.
import logging, json
logging.basicConfig(level=logging.WARNING)
from mrag.ask import init_pipeline

pipeline = init_pipeline(load_image_embedder=False, load_vlm=False)
assert type(pipeline.kg).__name__ == "VineKG", "retrieval is not on the VINE graph -- check CFG.graph_backend"

n_store = pipeline.store.client.count(CFG.coll_chunks, exact=True).count
print(f"graph        : {type(pipeline.kg).__name__}, {pipeline.kg.g.number_of_nodes():,} nodes, "
      f"{pipeline.kg.g.number_of_edges():,} edges")
print(f"vector store : {n_store:,} items")
SETTINGS = {k: getattr(CFG, k) for k in (
    "top_k_fused", "top_k_after_graph", "top_k_compile_chunks",
    "expansion_source_chunks", "expansion_reserved_slots", "compile_closure_budget")}
print("retrieval    :", SETTINGS)

In [ ]:
# CELL 4 — the questions.
# DEV: written from random MUTCD provisions, picked by a fixed rule (the first
#      scenario case of each Part, and the first two-section case of each kind).
# TEST: the 10 sample questions. TEXT ONLY -- no sections, answers, obligations or
#      counts. They are parsed once, in CELL 12. Nothing is changed because of them.
import json, time, importlib
from pathlib import Path
from mrag.config import CFG
import parser_check as pc
importlib.reload(pc)

DEV = pc.load_dev_cases()
print(len(DEV), "DEV cases:", [c["case_id"] for c in DEV])

TEST = {
 "SAMPLE002": "A conventional-road underpass has a measured minimum vertical clearance of 14 feet 7.4 inches. The statutory maximum vehicle height is 13 feet 9 inches. The location experiences frost action, and the agency has selected a 2-inch frost allowance. The clearance is uniform across the traveled way. Determine whether an advance low-clearance sign is required, the whole-inch clearance legend recommended after applying that allowance, and whether a panel mounted on the structure could replace the advance sign. If the structure panel is added, what legend should it show? Distinguish the Standard, Guidance, and Option provisions; no advance-placement distance is requested.",
 "SAMPLE003": "Use the lowest of the three high-visibility designs in Figure 3C-1. A legally established non-intersection crossing on a road posted at 40 mph is proposed with a crosswalk width of 7 feet, measured in the direction of approaching vehicle travel. Across the roadway, the design has four white longitudinal bars arranged as two identical pairs. Each bar is 10 inches wide, the clear gap inside each pair is 10 inches, and the clear gap between the two pairs is 70 inches. There are no transverse lines. The crossing location, warning signs, and other site treatments have been resolved separately. Identify the illustrated design and decide whether these specified marking dimensions and element count satisfy the applicable Standards. Give the minimum number of pairs and the permissible between-pair gap for the stated bar dimensions.\n\n**Required visual input:** [Figure 3C-1 — original PDF page 631](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=631). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE004": "Two entrance-ramp alternatives are being checked for signal-face count and placement only. In Alternative A, the ramp has one metered general-purpose lane and a non-metered HOV bypass lane. Two faces serve the metered lane, both on one roadside pole, with the lower face mounted 4.5 feet high. In Alternative B, both lanes are metered and their greens alternate rather than always appearing simultaneously. B provides exactly one face for each lane, each on the nearest roadside pole, with no face over either lane. Other signal design and visibility provisions have been satisfied. Can the count-and-placement arrangements be accepted in both alternatives? Explain how the controlled-lane count changes the applicable rule and identify a compliant correction to B.",
 "SAMPLE005": "A reversible-lane installation, away from any toll plaza, has minimal visual clutter and is a typical application. During some periods the lane serves left turns, and all required arrow and X indications have been provided. The nominal face dimensions are 12 by 12 inches for the downward green arrow, yellow X, and red X, and 18 by 18 inches for each applicable white left-turn arrow. Check only these face dimensions. In Operating Case A, the posted and statutory speed limits, 85th-percentile speed, and operating speed are all 35 mph. In Case B all four are 40 mph. Can the same small faces be accepted in both cases? Explain the exception boundary and the nominal minimum dimensions that apply when it is unavailable.",
 "SAMPLE006": "An ordinary signalized crosswalk has a concurrent vehicular phase, with an engineered pedestrian sequence of 7 seconds WALK, 12 seconds flashing UPRAISED HAND, and 3 seconds steady UPRAISED HAND as a buffer. No preemption occurs. A pedestrian enters the crosswalk 110 feet from the countdown display. The proposed display uses 6-inch Portland orange numbers on a black opaque background immediately adjacent to the hand indication. It stays dark during WALK, begins at 15 when the flashing hand starts, and continues counting through the 3-second steady-hand buffer before reaching zero. Are its countdown operation and numeral height acceptable under their respective authorities? State what time should be counted and whether the long viewing distance creates a mandatory or recommended larger numeral size. Do not recalculate the engineered pedestrian intervals.",
 "SAMPLE007": "A two-lane, one-way public highway crosses two railroad tracks at a passive crossing. There are no active warning devices and no authorized person routinely stopping road users when a train approaches. There is no nearby signalized highway intersection, and no engineering study has justified replacing the default YIELD control with STOP control. The proposal provides only a right-side Crossbuck with a 2 TRACKS plaque and a YIELD sign on a separate support. The YIELD sign positioning and height, Crossbuck blade retroreflection, and other placement details have already been checked. Both the front and back of the Crossbuck support are left without retroreflective strips, on the claim that one-way operation permits both omissions. Are the assembly count, control choice, and support-strip omissions acceptable? State what must change and which permission can still be used.",
 "SAMPLE008": "A fabricator is preparing a paddle for an adult school crossing guard who will use it before dawn. One face says STOP; the other copies the face shown in the middle row of the PREFERRED METHOD column in Figure 6D-1. The paddle is octagonal, 24 inches across, on a rigid handle, and fully retroreflectorized. All other size and legend details are appropriate to the face being copied. The STOP face has two flashing lights, one centered above its legend and one below: the upper light is white, the lower red, and they flash at 55 flashes per minute. Can this device be accepted for the school guard? Identify the copied face, follow the school-use cross-reference, and distinguish the permitted flashing-light feature from its mandatory restrictions.\n\n**Required visual input:** [Figure 6D-1 — original PDF page 825](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=825). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE009": "An emergency-vehicle hybrid beacon has been justified for an unsignalized fire-station access. All application criteria have been satisfied, no STOP- or YIELD-controlled side-road intersection lies within 100 feet, and no grade crossing is involved. It is actuated only by authorized personnel. On each major-street approach, all relevant speed measures are 45 mph. There are two beacon faces per approach, both roadside and none over the roadway; required signs and stop lines are provided. The normal program is dark, flashing yellow, steady yellow, alternating flashing red for sufficient vehicle egress, then dark again. The yellow timing has been properly engineered. If the conflict monitor puts the beacon into flashing mode, the proposed fault program also displays alternating flashing red. Assess face count, recommended placement, normal operation, and fault operation. Which change is mandatory and which is a Guidance matter?",
 "SAMPLE010": "A three-lane, one-way street has a marked midblock crosswalk across an uncontrolled approach. An engineering study supports warning lights embedded in the pavement, applicable warning signs are installed, and no STOP, YIELD, traffic signal, or pedestrian hybrid beacon controls the crossing. The approach-side installation extends across the full crossing and provides three unidirectional units, one per lane. Each is 9 feet from the outside edge of the nearest crosswalk line and faces away from the crosswalk toward approaching traffic. No units are proposed on the departure side. Passive pedestrian detection starts flashing and ends operation after the pedestrian clears the full crosswalk. Yellow output has 55 flash periods per minute; within those periods, its separate flicker modulation repeats at 4 pulses per second. Units project 0.5 inch above the pavement. Evaluate only side-of-crosswalk placement, minimum count, offset, facing direction, actuation, flash/pulse rates, and height. Can those specified features be accepted, and which are governed by an Option or Guidance rather than a Standard?",
 "SAMPLE011": "A continuous paved, undivided, two-way arterial has exactly two traffic lanes, a 20-foot traveled way, and an ADT of 6,500 vehicles. One segment is classified rural and the adjoining segment urban. It is neither a freeway nor an expressway. An engineering study also identifies a need for edge lines on the urban segment. There is no finding that edge lines would decrease safety, and no edge-line exclusion or omission is being used. Considering only whether center lines and edge lines are called for, identify which are required by a Standard and which are recommended by Guidance on each segment. A reviewer claims that unchanged width and traffic volume make the authority identical on both segments. Is that claim correct?"
}
print(len(TEST), "TEST questions (text only)")

In [ ]:
# CELL 5 — retrieval for every question, saved to Drive.
# The parser reads exactly this. Saved once per commit, so every later run (and
# every restart) parses the same input.
import time, numbers
RUNS = Path(CFG.base_dir) / "parser_runs"
RUNS.mkdir(parents=True, exist_ok=True)
KQ_FILE = RUNS / f"kq_{COMMIT}.json"
QUESTIONS = {**{c["case_id"]: c["text"] for c in DEV}, **TEST}

def plain(x):
    # numpy numbers and anything else odd -> plain JSON values
    if isinstance(x, dict):
        return {str(k): plain(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [plain(v) for v in x]
    if isinstance(x, bool) or x is None or isinstance(x, str):
        return x
    if isinstance(x, numbers.Integral):
        return int(x)
    if isinstance(x, numbers.Real):
        return float(x)
    return str(x)

FIG_KEYS = ("figure_id", "caption", "n_sheets", "image_paths", "image_path", "page_pdf", "source")
if not KQ_FILE.exists() and "pipeline" not in globals():
    raise RuntimeError(f"retrieval for commit {COMMIT} is not saved yet ({KQ_FILE.name}). "
                       "Run CELL 2 and CELL 3 first, then this cell.")
if KQ_FILE.exists():
    KQ = json.loads(KQ_FILE.read_text())
    missing = [q for q in QUESTIONS if q not in KQ]
    assert not missing, f"{KQ_FILE.name} has no retrieval for {missing}; delete the file and re-run CELL 2, 3, 5"
    print(f"read {KQ_FILE.name}: {len(KQ)} questions")
else:
    KQ = {}
    for i, (qid, q) in enumerate(QUESTIONS.items(), 1):
        t0 = time.time()
        res = pipeline.retriever.retrieve_for_compile(q)
        KQ[qid] = plain({"question": q, "chunks": res.chunks,
                         "figures": [{k: f.get(k) for k in FIG_KEYS if k in f} for f in res.figures],
                         "closure": res.debug.get("closure", {}),
                         "seconds": round(time.time() - t0, 1)})
        print(f"  {i:2d}/{len(QUESTIONS)} {qid:26s} Kq {len(res.chunks):3d} items, "
              f"{len(res.figures)} figures, {time.time() - t0:4.1f}s")
    KQ_FILE.write_text(json.dumps(KQ))
    print("saved ->", KQ_FILE)

for c in DEV:
    t = pc.target_check(None, c, KQ[c["case_id"]]["chunks"])
    print(f"  DEV {c['case_id']:26s} target provision reaches the parser: {t['target_chunk_in_kq']}")

In [ ]:
# CELL 6 — free the GPU for the model; load what the checks need.
# The checks read the VINE graph (does a section / figure / table exist?) and the
# table transcriptions (is a column one the table prints?). Both are CPU only.
import gc, torch
import mrag.ask as _mask
from mrag.kg_vine import VineKG
from mrag.vine import table_data
from mrag.vine import vllm_client as vc

vc.stop_servers()        # a model server left from an earlier run still holds the GPU

if "pipeline" in globals():
    kg = pipeline.kg
    del pipeline
    _mask._pipeline = None
elif "kg" not in globals():
    kg = VineKG(CFG.vine_graph, figures_path=CFG.figures_jsonl, sign_codes_path=CFG.sign_codes_json,
                items_path=CFG.vine_items, tables_path=CFG.tables_jsonl,
                table_rows_in_closure=getattr(CFG, "closure_table_rows", 15))
gc.collect(); torch.cuda.empty_cache()

TABLES = {}
for t in table_data.load(Path(CFG.tables_jsonl)):
    TABLES.setdefault(t.table_id, []).append(t)

free, total = torch.cuda.mem_get_info()
name = torch.cuda.get_device_name(0)
print(f"GPU          : {name}, {total / 2**30:.1f} GiB, free {free / 2**30:.1f} GiB")
print(f"graph checks : section 1A.01 known = {kg.is_known_citation('section', '1A.01')}")
print(f"tables       : {len(TABLES)} table ids with transcriptions")
assert total / 2**30 > 38, ("this model needs an A100 (40 or 80 GB) or larger. "
                            "Runtime > Change runtime type > A100, then run CELL 1, 4, 5, 6 again.")
GPU_UTIL = 0.90
assert free >= GPU_UTIL * total, (
    f"only {free / 2**30:.1f} GiB free; vLLM needs {GPU_UTIL * total / 2**30:.1f}. "
    "Restart the runtime, then run CELL 1, 4, 5, 6 (retrieval is saved).")

In [ ]:
# CELL 7 — vLLM in its own environment, and the model on local disk.
# Its own environment so its newer torch/transformers never touch the retrieval
# stack above. About 5 minutes to install and 5-10 to download 28 GB.
VENV          = "/content/vllm_env"
VLLM_VERSION  = "0.31.0"
MODEL_ID      = "Qwen/Qwen3.8-27B-FP8"      # 27.8 GB; on an A100 vLLM runs it as 8-bit weights
HF_LOCAL      = "/content/hf_local"          # local disk, not Drive: faster, and Drive stays clean

# mrag's config points the Hugging Face cache at Drive with HUGGINGFACE_HUB_CACHE,
# and that older variable wins over HF_HOME. Every cache variable is set to local
# disk here, so the download and the server both use /content/hf_local.
for k in ("HF_HOME", "HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"):
    os.environ.pop(k, None)
os.environ["HF_HOME"] = HF_LOCAL
os.environ["HF_HUB_CACHE"] = f"{HF_LOCAL}/hub"

if not os.path.exists(f"{VENV}/bin/vllm"):
    !pip install -q uv
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python "vllm=={VLLM_VERSION}" --torch-backend=auto
!{VENV}/bin/python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| transformers', transformers.__version__)"
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

!{VENV}/bin/python -c "from huggingface_hub import snapshot_download as s; print(s('{MODEL_ID}'))"
SNAPS = Path(HF_LOCAL) / "hub" / ("models--" + MODEL_ID.replace("/", "--")) / "snapshots"
assert SNAPS.is_dir(), f"the model did not land in {SNAPS} -- send the output above"
MODEL_PATH = sorted(SNAPS.iterdir(), key=lambda p: p.stat().st_mtime)[-1]
REVISION = MODEL_PATH.name
print("model on local disk:", MODEL_PATH)
print("model revision     :", REVISION)
!df -h /content | tail -1

In [ ]:
# CELL 8 — start the model server and check it answers.
# One request at a time, fixed seed, the model card's thinking-mode sampling.
# First start takes 5-15 minutes (loading, compiling, CUDA graphs).
import hashlib
from mrag.vine import vllm_client as vc
from mrag.vine.parser import PARSER_TASK, PARSER_CONTRACT

SEED, EFFORT, MAX_TOKENS, MAX_LEN = 42, "medium", 20480, 49152
SERVER = vc.launch(f"{VENV}/bin/vllm", str(MODEL_PATH), log_path="/content/vllm_server.log", port=8000,
                   env={"HF_HUB_OFFLINE": "1",
                        # FlashInfer's sampler is compiled on first use and that
                        # fails on this setup. Seeded requests never use it anyway.
                        "VLLM_USE_FLASHINFER_SAMPLER": "0"},
                   served_name="parser", max_model_len=MAX_LEN,
                   gpu_memory_utilization=GPU_UTIL, seed=SEED)
print(f"server ready in {SERVER['seconds']:.0f}s")

ENGINE = f"vllm {VLLM_VERSION}"
probe = vc.make_ask_vllm(SERVER["base_url"], "parser", model_id=MODEL_ID, revision=REVISION,
                         engine=ENGINE, seed=SEED, max_tokens=2048, reasoning_effort="low")
reply = probe('Reply with this JSON object and nothing else: {"ok": true}', [])
u = probe.last["usage"]
print("probe reply  :", repr(reply[:80]))
print(f"probe        : {u.get('completion_tokens')} tokens in {probe.last['seconds']}s, "
      f"finish_reason {probe.last['finish_reason']}, "
      f"thinking {probe.last['reasoning_chars']} chars kept out of the reply")
assert '"ok"' in reply, "the server answered, but not with the JSON asked for -- send the output above"

ask = vc.make_ask_vllm(SERVER["base_url"], "parser", model_id=MODEL_ID, revision=REVISION,
                       engine=ENGINE, seed=SEED, max_tokens=MAX_TOKENS, reasoning_effort=EFFORT,
                       cache_dir=str(RUNS / "model_cache"))
RUN_DIR = RUNS / time.strftime("run_%Y%m%d_%H%M")
RUN_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = {"commit": COMMIT, "model": MODEL_ID, "revision": REVISION, "vllm": VLLM_VERSION,
            "server_command": SERVER["cmd"], "gpu": torch.cuda.get_device_name(0), "seed": SEED,
            "sampling": vc.QWEN_THINKING, "reasoning_effort": EFFORT, "max_tokens": MAX_TOKENS,
            "max_model_len": MAX_LEN, "max_attempts": 3, "fallback": False,
            "instructions_sha256": hashlib.sha256((PARSER_TASK + PARSER_CONTRACT).encode()).hexdigest(),
            "kq_file": KQ_FILE.name, "retrieval_settings": SETTINGS if "SETTINGS" in globals() else None}
(RUN_DIR / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))
print("run folder   :", RUN_DIR)

### CELL 9 parses one DEV case. Read the plan before going on.

Things to look for:
- **outcome** — `valid first time` is best; `valid after repair` means the checks sent it back and it fixed the problems; `failed` means 3 tries did not pass.
- **obligations** — each should come from one provision (`<-` id) and be one check. Not one per retrieved paragraph.
- **requires / guard / merges** — the structure. A flat list with no dependencies and no merges is the old deterministic compiler's shape.
- **numbers found in no input** — a number in a claim that is in neither the question nor the retrieved text. The model made it up or worked it out; both are wrong for a parser.
- **model calls** — `finish_reason` should be `stop`. `length` means the model's thinking ran out of room before it wrote the plan; send me the output.

In [ ]:
# CELL 9 — one DEV case.
from mrag.vine.parser import make_semantic_parser
from mrag.vine.compile import instantiate

parse = make_semantic_parser(ask, max_attempts=3, fall_back_to_baseline=False,
                             check_ref=kg.is_known_citation, tables=TABLES)

def run_one(qid, case=None):
    v = KQ[qid]
    t0 = time.time()
    spec, rep = parse(v["question"], v["chunks"], "", v["figures"])
    net = instantiate(spec)[0] if spec is not None else None
    return pc.make_record(qid, v["question"], v, spec, rep, net, time.time() - t0, case=case)

first = DEV[0]
print(first["text"], "\n")
rec = run_one(first["case_id"], first)
pc.print_plan(rec)
print("\nmodel calls:", [(c["cached"], c["finish_reason"], (c["usage"] or {}).get("completion_tokens"), c["seconds"]) for c in rec["calls"]])
if rec["target"]["target_chunk_in_kq"]:
    print("plan uses the provision this case was written from:", rec["target"]["source_hit"])

In [ ]:
# CELL 10 — every DEV case (about 30-50 minutes), then the summary.
DEV_RECORDS = []
for i, c in enumerate(DEV, 1):
    r = run_one(c["case_id"], c)
    DEV_RECORDS.append(r)
    s = r["stats"]
    print(f"  {i:2d}/{len(DEV)} {c['case_id']:26s} {r['outcome']:19s} attempts {r['attempts']} "
          f"obligations {s.get('obligations', 0):2d} deps {s.get('dependencies', 0):2d} "
          f"merges {s.get('merges', 0)} | {r['seconds']:5.0f}s")
(RUN_DIR / "dev_plans.json").write_text(json.dumps(DEV_RECORDS, indent=1, default=str))
print()
pc.print_summary(pc.summarize(DEV_RECORDS))
print("\nsaved ->", RUN_DIR / "dev_plans.json", "\n")
for r in DEV_RECORDS:
    pc.print_plan(r)
    print()

In [ ]:
# CELL 11 — does the same input give the same plan? Three DEV cases again,
# this time NOT from the cache: the model is asked afresh with the same seed.
fresh = vc.make_ask_vllm(SERVER["base_url"], "parser", model_id=MODEL_ID, revision=REVISION,
                         engine=ENGINE, seed=SEED, max_tokens=MAX_TOKENS, reasoning_effort=EFFORT,
                         cache_dir=None)
parse_fresh = make_semantic_parser(fresh, max_attempts=3, fall_back_to_baseline=False,
                                   check_ref=kg.is_known_citation, tables=TABLES)
same, tried = 0, 0
for r in [r for r in DEV_RECORDS if r["spec"] is not None][:3]:
    v = KQ[r["case_id"]]
    again, _rep = parse_fresh(v["question"], v["chunks"], "", v["figures"])
    a = json.dumps(r["spec"], sort_keys=True)
    b = json.dumps(again.as_dict() if again is not None else None, sort_keys=True)
    same += a == b
    tried += 1
    print(f"  {r['case_id']:26s} identical plan on a fresh run: {a == b}")
print(f"\n  {same}/{tried} identical")
MANIFEST["repeat_check"] = f"{same}/{tried} identical"
(RUN_DIR / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))

### CELL 12 — the TEST questions, once

Run this **once**, after CELL 10 and 11. Nothing in the pipeline is to be changed because of what it shows.
The plans are saved to Drive and downloaded, for scoring outside the pipeline. There are no gold answers in this notebook.

In [ ]:
# CELL 12 — the 10 TEST questions, once (about 30-50 minutes).
TEST_RECORDS = []
for i, qid in enumerate(TEST, 1):
    r = run_one(qid)
    TEST_RECORDS.append(r)
    s = r["stats"]
    print(f"  {i:2d}/{len(TEST)} {qid:10s} {r['outcome']:19s} attempts {r['attempts']} "
          f"obligations {s.get('obligations', 0):2d} deps {s.get('dependencies', 0):2d} "
          f"merges {s.get('merges', 0)} | {r['seconds']:5.0f}s")
out = RUN_DIR / "test_plans.json"
out.write_text(json.dumps(TEST_RECORDS, indent=1, default=str))
print()
pc.print_summary(pc.summarize(TEST_RECORDS))
print("\nsaved ->", out, "\n")
for r in TEST_RECORDS:
    pc.print_plan(r)
    print()
from google.colab import files
files.download(str(out))